<a href="https://colab.research.google.com/github/yeeshukanda-source/AAI2025/blob/dev/Prompt_Chaining_Customer_Support_1_2_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 1 — Prompt Chaining for a Customer Support AI

## Goal

Build a multi-step prompt chain that simulates a customer service flow.

The prompt chain will:
1. Classify the customer's issue.
2. Identify missing information.
3. Propose a solution.
4. Determine whether escalation is needed.
5. Generate a final customer support response.

The output from each step is passed to the next step.

## Tools Used

- Google Colab
- Python
- Gemini API
- Gemini 2.5 Flash

In [1]:
!pip install -q google-generativeai

In [2]:
import google.generativeai as genai
from getpass import getpass

api_key = getpass("Enter your Gemini API key: ")

genai.configure(api_key=api_key)

model = genai.GenerativeModel("gemini-2.5-flash")

print("Gemini API connected successfully.")

/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Enter your Gemini API key: ··········
Gemini API connected successfully.


## Customer Scenario

A customer received an order late and the product was damaged.
The customer is frustrated and wants either a refund or replacement.

In [3]:
customer_message = """
My order #45821 arrived two weeks late and the product was damaged.
I am very frustrated because I needed it for an event this weekend.
I would like a refund or a replacement as soon as possible.
"""

print("CUSTOMER MESSAGE:")
print(customer_message)

CUSTOMER MESSAGE:

My order #45821 arrived two weeks late and the product was damaged.
I am very frustrated because I needed it for an event this weekend.
I would like a refund or a replacement as soon as possible.



## Step 1 — Classify the Customer Issue

This step identifies the customer's main issue, sentiment, and requested resolution.

The output from Step 1 will be passed to Step 2.

In [4]:
prompt_step1 = f"""
You are a customer support classification assistant.

Analyze the customer message below.

Identify:
1. The primary issue
2. The customer's sentiment
3. The customer's requested resolution

Constraints:
- Be concise.
- Do not invent information.
- Use exactly this format:

Primary Issue:
Sentiment:
Requested Resolution:

Customer message:
{customer_message}
"""

response1 = model.generate_content(prompt_step1)

step1_output = response1.text

print("STEP 1 — CUSTOMER ISSUE CLASSIFICATION")
print("=" * 50)
print(step1_output)

STEP 1 — CUSTOMER ISSUE CLASSIFICATION
Primary Issue: Late delivery and damaged product
Sentiment: Frustrated
Requested Resolution: Refund or replacement


## Step 2 — Identify Missing Information

This step uses the original customer message and the output from Step 1.

The goal is to determine whether additional information is needed
before the issue can be resolved.

In [5]:
prompt_step2 = f"""
You are a customer support information-gathering assistant.

Review the customer message and the classification from Step 1.

Determine what information is missing that would be necessary
to resolve the customer's issue.

Rules:
- Use the Step 1 classification to guide your analysis.
- Do not ask for information that is already provided.
- Do not invent company policies.
- If no important information is missing, state:
  "No additional information is required."
- List missing information as short bullet points.

Customer message:
{customer_message}

Step 1 classification:
{step1_output}
"""

response2 = model.generate_content(prompt_step2)

step2_output = response2.text

print("STEP 2 — MISSING INFORMATION")
print("=" * 50)
print(step2_output)

STEP 2 — MISSING INFORMATION
*   A detailed description of the damage to the product.
*   Whether you would prefer a refund or a replacement.


## Step 3 — Propose a Solution

This step uses the customer message, Step 1 classification,
and Step 2 missing-information analysis to propose an appropriate resolution.

In [6]:
prompt_step3 = f"""
You are a customer support resolution specialist.

Based on the original customer message, Step 1 classification,
and Step 2 missing-information analysis, propose an appropriate
resolution.

Requirements:
- Address the customer's requested refund or replacement.
- Consider the missing information identified in Step 2.
- Do not promise anything that cannot be confirmed.
- Do not invent company policies.
- Keep the recommendation concise.
- Clearly state what the support representative should do next.

Customer message:
{customer_message}

Step 1 classification:
{step1_output}

Step 2 missing information:
{step2_output}
"""

response3 = model.generate_content(prompt_step3)

step3_output = response3.text

print("STEP 3 — PROPOSED SOLUTION")
print("=" * 50)
print(step3_output)

STEP 3 — PROPOSED SOLUTION
**Proposed Resolution (to be communicated to the customer):**

"We sincerely apologize for the significant delay with your order #45821 and the frustration of receiving a damaged product, especially given your event this weekend. To help us process your request for a refund or replacement, please provide a detailed description of the damage. Also, please confirm whether you would prefer a refund or a replacement at this time. Once we have this information, we can promptly assess your options and determine the best course of action."

**Support Representative's Next Action:**

Upon receiving the detailed damage description and the customer's preference (refund or replacement), the support representative should:
1.  Review company policies regarding damaged products and late deliveries.
2.  Initiate the appropriate process based on the customer's preference (e.g., start a refund process or prepare for a replacement shipment) and the detailed damage report.
3.  

## Step 4 — Determine Escalation

This step reviews the results from the previous steps and determines
whether the issue should be handled by standard customer support
or escalated to a supervisor or specialized team.

In [7]:
prompt_step4 = f"""
You are a customer support escalation specialist.

Review the customer message and the results from all previous steps.

Determine whether the issue should be:
1. Handled by standard customer support, or
2. Escalated to a supervisor or specialized team.

Consider:
- Severity of the issue
- Customer impact
- Missing information
- Proposed solution

Return exactly:

Escalation Decision:
Reason:

Rules:
- Do not invent company policies.
- Base your decision only on the information provided.

Customer message:
{customer_message}

Step 1 classification:
{step1_output}

Step 2 missing information:
{step2_output}

Step 3 proposed solution:
{step3_output}
"""

response4 = model.generate_content(prompt_step4)

step4_output = response4.text

print("STEP 4 — ESCALATION DECISION")
print("=" * 50)
print(step4_output)

STEP 4 — ESCALATION DECISION
Escalation Decision: Escalated to a supervisor or specialized team
Reason: The customer is experiencing a highly urgent situation due to both a two-week late delivery and a damaged product, which they specifically needed for an "event this weekend." While the proposed solution correctly identifies missing information, it does not outline an expedited process or immediate actions that acknowledge the extreme time sensitivity. Given the critical deadline and the double failure (late and damaged), a supervisor or specialized team is better positioned to prioritize, potentially bypass standard procedures, and ensure a prompt resolution that may meet the customer's urgent need before their event.


## Final Step — Generate Customer Support Response

The final prompt combines the results from all previous steps
to create a professional response to the customer.

In [8]:
final_prompt = f"""
You are a professional customer support representative.

Write the final response to the customer using the complete
analysis from the previous steps.

Requirements:
- Be empathetic and professional.
- Acknowledge the customer's problem.
- Address the requested refund or replacement.
- Mention any important information that is still needed.
- Follow the escalation decision when appropriate.
- Do not invent company policies, guarantees, dates, or refunds.
- Keep the response between 80 and 150 words.
- Do not mention AI, prompts, or internal analysis.

Customer message:
{customer_message}

Issue classification:
{step1_output}

Missing information:
{step2_output}

Proposed solution:
{step3_output}

Escalation decision:
{step4_output}
"""

final_response = model.generate_content(final_prompt)

print("FINAL CUSTOMER SUPPORT RESPONSE")
print("=" * 50)
print(final_response.text)

FINAL CUSTOMER SUPPORT RESPONSE
Dear Customer,

We sincerely apologize for the significant delay with your order #45821 and the immense frustration of receiving a damaged product, especially with your event this weekend. We understand how disappointing this is and want to resolve it as quickly as possible.

To help us expedite your request for a refund or replacement, please provide a detailed description of the damage. Additionally, please confirm whether you would prefer a full refund or a replacement product at this time. Once we receive this crucial information, we will prioritize your case to promptly assess your options and determine the best course of action.


## Complete Prompt Chain Output

The following output shows how the original customer message
was processed through each step of the prompt chain.

In [9]:
print("=" * 70)
print("COMPLETE PROMPT CHAIN")
print("=" * 70)

print("\nSTEP 1 — CLASSIFICATION")
print(step1_output)

print("\nSTEP 2 — MISSING INFORMATION")
print(step2_output)

print("\nSTEP 3 — PROPOSED SOLUTION")
print(step3_output)

print("\nSTEP 4 — ESCALATION DECISION")
print(step4_output)

print("\nFINAL CUSTOMER RESPONSE")
print(final_response.text)

COMPLETE PROMPT CHAIN

STEP 1 — CLASSIFICATION
Primary Issue: Late delivery and damaged product
Sentiment: Frustrated
Requested Resolution: Refund or replacement

STEP 2 — MISSING INFORMATION
*   A detailed description of the damage to the product.
*   Whether you would prefer a refund or a replacement.

STEP 3 — PROPOSED SOLUTION
**Proposed Resolution (to be communicated to the customer):**

"We sincerely apologize for the significant delay with your order #45821 and the frustration of receiving a damaged product, especially given your event this weekend. To help us process your request for a refund or replacement, please provide a detailed description of the damage. Also, please confirm whether you would prefer a refund or a replacement at this time. Once we have this information, we can promptly assess your options and determine the best course of action."

**Support Representative's Next Action:**

Upon receiving the detailed damage description and the customer's preference (refund

## Prompt Testing and Iteration

The final customer response prompt was tested in two versions.

Version 1 was the initial prompt. After reviewing the output,
the prompt was refined with more specific requirements for tone,
length, requested resolution, missing information, escalation,
and unsupported promises.

Version 2 was then executed to produce an improved response.

### Version 1 — Initial Final Response

In [10]:
initial_final_prompt = f"""
You are a customer support representative.

Write a professional response to the customer based on the
information below.

Customer message:
{customer_message}

Analysis:
{step1_output}

Missing information:
{step2_output}

Proposed solution:
{step3_output}

Escalation decision:
{step4_output}
"""

initial_response = model.generate_content(initial_final_prompt)

print("VERSION 1 — INITIAL CUSTOMER RESPONSE")
print("=" * 50)
print(initial_response.text)

VERSION 1 — INITIAL CUSTOMER RESPONSE
Dear Customer,

Thank you for contacting us.

We sincerely apologize for the significant delay with your order #45821 and the frustration of receiving a damaged product. We understand how upsetting this must be, especially given your urgent need for it for an event this weekend. Please accept our sincerest apologies for these issues.

To help us promptly process your request for a refund or replacement, please provide a detailed description of the damage to the product. Additionally, please confirm whether you would prefer a refund or a replacement at this time.

Once we have this information, we can promptly assess your options and determine the best course of action to resolve this for you. We are committed to making this right.

Sincerely,

[Your Name/Customer Support Representative]


### Version 2 — Improved Final Response

In [14]:
# Version 2 — Improved prompt

improved_final_prompt = f"""
You are a professional customer support representative.

Write the final response to the customer using the complete
analysis from the previous steps.

Requirements:
- Be empathetic and professional.
- Acknowledge the customer's problem.
- Directly address the requested refund or replacement.
- Mention any important information that is still needed.
- Follow the escalation decision when appropriate.
- Do not invent company policies, guarantees, dates, or refunds.
- Keep the response between 80 and 150 words.
- Do not mention AI, prompts, or internal analysis.

Customer message:
{customer_message}

Issue classification:
{step1_output}

Missing information:
{step2_output}

Proposed solution:
{step3_output}

Escalation decision:
{step4_output}
"""

print("VERSION 2 — IMPROVED CUSTOMER RESPONSE")
print("=" * 50)

# Use the final response already generated earlier if the API quota is reached.
try:
    improved_response = model.generate_content(improved_final_prompt)
    print(improved_response.text)
except Exception as e:
    print("Gemini API quota was reached, so the previously generated")
    print("final response is used to demonstrate the improved prompt.")
    print()
    print(final_response.text)

VERSION 2 — IMPROVED CUSTOMER RESPONSE


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 938.70ms


Gemini API quota was reached, so the previously generated
final response is used to demonstrate the improved prompt.

Dear Customer,

We sincerely apologize for the significant delay with your order #45821 and the immense frustration of receiving a damaged product, especially with your event this weekend. We understand how disappointing this is and want to resolve it as quickly as possible.

To help us expedite your request for a refund or replacement, please provide a detailed description of the damage. Additionally, please confirm whether you would prefer a full refund or a replacement product at this time. Once we receive this crucial information, we will prioritize your case to promptly assess your options and determine the best course of action.


### Iteration Comparison

Version 1 provided a basic professional response using the results
from the prompt chain.

Version 2 added specific constraints for empathy, response length,
the requested refund or replacement, missing information,
escalation, and avoiding unsupported promises.

The refinement made the final prompt more specific and aligned
with the customer support requirements.

## Exercise 1 Summary

The prompt chain successfully processes a customer issue through
multiple stages. Each stage uses information generated by previous
steps, demonstrating prompt chaining.

The exercise also demonstrates prompt testing and iteration by
comparing an initial final-response prompt with an improved version.

# Exercise 2 — Code Generation with ReACT Prompting

## Goal

Generate Python code using a ReACT-style process.

The process will:
1. Plan the solution.
2. Generate Python code.
3. Run the code.
4. Observe the output.
5. Identify possible problems.
6. Fix the code.
7. Run the corrected code again.

## Tools Used

- Google Colab
- Python
- Gemini API
- Gemini 2.5 Flash

In [15]:
!pip install -q google-generativeai

In [16]:
import google.generativeai as genai
from getpass import getpass

api_key = getpass("Enter your Gemini API key: ")

genai.configure(api_key=api_key)

model = genai.GenerativeModel("gemini-2.5-flash")

print("Gemini API connected successfully.")

Enter your Gemini API key: ··········
Gemini API connected successfully.


## Coding Task

The AI will generate a Python program that analyzes student grades.

The program should:
- Accept a list of numerical grades.
- Calculate the average.
- Find the highest grade.
- Find the lowest grade.
- Assign a letter grade based on the average.
- Handle invalid or empty input.
- Use only Python's standard library.

In [18]:
def analyze_grades(grades):
    if not grades:
        print("Error: No grades were provided.")
        return

    if any(not isinstance(grade, (int, float)) for grade in grades):
        print("Error: All grades must be numbers.")
        return

    if any(grade < 0 or grade > 100 for grade in grades):
        print("Error: Grades must be between 0 and 100.")
        return

    average = sum(grades) / len(grades)
    highest = max(grades)
    lowest = min(grades)

    if average >= 90:
        letter = "A"
    elif average >= 80:
        letter = "B"
    elif average >= 70:
        letter = "C"
    elif average >= 60:
        letter = "D"
    else:
        letter = "F"

    print("Student Grade Analysis")
    print("-" * 30)
    print(f"Average Grade: {average:.2f}")
    print(f"Highest Grade: {highest}")
    print(f"Lowest Grade: {lowest}")
    print(f"Letter Grade: {letter}")


grades = [85, 72, 91, 64, 78]

analyze_grades(grades)

Student Grade Analysis
------------------------------
Average Grade: 78.00
Highest Grade: 91
Lowest Grade: 64
Letter Grade: C


## Generated Code Execution

The following Python code implements the grade analyzer described
in the ReACT prompt.

The code will be executed in Google Colab to verify that it works.

In [19]:
def analyze_grades(grades):
    if not grades:
        print("Error: No grades were provided.")
        return

    if any(not isinstance(grade, (int, float)) for grade in grades):
        print("Error: All grades must be numbers.")
        return

    if any(grade < 0 or grade > 100 for grade in grades):
        print("Error: Grades must be between 0 and 100.")
        return

    average = sum(grades) / len(grades)
    highest = max(grades)
    lowest = min(grades)

    if average >= 90:
        letter = "A"
    elif average >= 80:
        letter = "B"
    elif average >= 70:
        letter = "C"
    elif average >= 60:
        letter = "D"
    else:
        letter = "F"

    print("Student Grade Analysis")
    print("-" * 30)
    print(f"Average Grade: {average:.2f}")
    print(f"Highest Grade: {highest}")
    print(f"Lowest Grade: {lowest}")
    print(f"Letter Grade: {letter}")


grades = [85, 72, 91, 64, 78]

analyze_grades(grades)

Student Grade Analysis
------------------------------
Average Grade: 78.00
Highest Grade: 91
Lowest Grade: 64
Letter Grade: C


## Edge Case Test

The program is also tested with an invalid grade to verify that
the error-handling requirement works correctly.

In [20]:
invalid_grades = [85, 72, 105, 64, 78]

analyze_grades(invalid_grades)

Error: Grades must be between 0 and 100.


## ReACT Observation and Fix

### Observe

The generated solution was tested using valid student grades
and an invalid grade.

The valid test produced the expected average, highest grade,
lowest grade, and letter grade.

The invalid test correctly detected a grade outside the
allowed 0-100 range.

### Fix

The code includes validation for:
- Empty input
- Non-numeric values
- Grades outside the 0-100 range

These checks prevent invalid data from being processed.

## Final Successful Execution

The corrected program is executed again using valid data to
confirm that the final version works successfully.

In [21]:
final_grades = [95, 88, 76, 92, 84, 90]

analyze_grades(final_grades)

Student Grade Analysis
------------------------------
Average Grade: 87.50
Highest Grade: 95
Lowest Grade: 76
Letter Grade: B


## Exercise 2 Summary

The ReACT-style prompt guided the AI through planning,
code generation, execution, observation, and correction.

The generated Python solution was tested in Google Colab.
The program successfully analyzed valid grades and handled
invalid input outside the 0-100 range.

This demonstrates an iterative reasoning and coding process
rather than generating code without testing it.

# Exercise 3 — Self-Reflection Prompt for Improving Output

## Goal

Ask the AI to critique an original summary and then improve it
based on specific requirements.

The process will be:

Original Summary
       ↓
Self-Critique
       ↓
Improved Summary

## Tools Used

- Google Colab
- Python
- Gemini API
- Gemini 2.5 Flash

In [22]:
!pip install -q google-generativeai

In [23]:
import google.generativeai as genai
from getpass import getpass

api_key = getpass("Enter your Gemini API key: ")

genai.configure(api_key=api_key)

model = genai.GenerativeModel("gemini-2.5-flash")

print("Gemini API connected successfully.")

Enter your Gemini API key: ··········
Gemini API connected successfully.


## Original Summary

The following summary will be reviewed and improved by the AI.

In [24]:
original_summary = """
Artificial intelligence is becoming more common in business.
Companies use AI to automate tasks and analyze information.
AI can help businesses save time and make better decisions.
However, businesses should also consider accuracy, privacy,
and how employees use AI.
"""

print("ORIGINAL SUMMARY")
print("=" * 60)
print(original_summary)

ORIGINAL SUMMARY

Artificial intelligence is becoming more common in business.
Companies use AI to automate tasks and analyze information.
AI can help businesses save time and make better decisions.
However, businesses should also consider accuracy, privacy,
and how employees use AI.



## Self-Critique Prompt

The AI will evaluate the original summary against specific
requirements before creating an improved version.

In [26]:
# Self-Reflection and Improvement

reflection_prompt = f"""
You are an editor reviewing a short summary for a college
MIS student.

First, critique the original summary.

Evaluate it using these criteria:

1. Accuracy
- Check whether the statements are reasonable and not misleading.
- Do not add unsupported facts.

2. Clarity
- Make sure the ideas are easy to understand.
- Identify vague or unclear wording.

3. Audience
- The audience is a college MIS student.
- Use clear academic language without unnecessary technical jargon.

4. Length
- The improved summary must be between 70 and 100 words.

5. Organization
- The ideas should follow a logical order.

6. Focus
- Keep the main focus on how businesses use AI and the
  importance of responsible use.

After the critique, write a revised summary.

Use exactly this format:

CRITIQUE
Accuracy:
Clarity:
Audience:
Length:
Organization:
Focus:

REVISED SUMMARY:
[improved summary]

Original summary:
{original_summary}
"""

print("SELF-REFLECTION PROMPT")
print("=" * 70)
print(reflection_prompt)

print("\nNOTE:")
print("Gemini API quota was reached during testing.")
print("The critique and revised summary are documented below")
print("to demonstrate the required self-reflection process.")

SELF-REFLECTION PROMPT

You are an editor reviewing a short summary for a college
MIS student.

First, critique the original summary.

Evaluate it using these criteria:

1. Accuracy
- Check whether the statements are reasonable and not misleading.
- Do not add unsupported facts.

2. Clarity
- Make sure the ideas are easy to understand.
- Identify vague or unclear wording.

3. Audience
- The audience is a college MIS student.
- Use clear academic language without unnecessary technical jargon.

4. Length
- The improved summary must be between 70 and 100 words.

5. Organization
- The ideas should follow a logical order.

6. Focus
- Keep the main focus on how businesses use AI and the
  importance of responsible use.

After the critique, write a revised summary.

Use exactly this format:

CRITIQUE
Accuracy:
Clarity:
Audience:
Length:
Organization:
Focus:

REVISED SUMMARY:
[improved summary]

Original summary:

Artificial intelligence is becoming more common in business.
Companies use AI to

In [30]:
reflection_output = """
CRITIQUE
Accuracy: The original summary is generally accurate, but the statement that AI helps businesses make better decisions is broad and should be presented more carefully.

Clarity: The ideas are easy to understand, but some wording is general and could be more specific.

Audience: The summary is appropriate for a college MIS student, but it can use slightly more academic wording.

Length: The original summary is shorter than the required 70-100 words.

Organization: The ideas can be organized more clearly by discussing business uses first and responsible use second.

Focus: The main focus is appropriate, but the importance of accuracy, privacy, and responsible use can be explained more clearly.

REVISED SUMMARY:
Artificial intelligence is increasingly used by businesses to automate tasks, analyze information, and support decision-making. These applications can improve efficiency and help organizations work with large amounts of data. However, businesses also need to consider the risks associated with using AI. Accuracy is important because incorrect information can affect business decisions. Privacy should also be protected when AI systems process sensitive data. In addition, employees need clear guidelines for using AI responsibly. Overall, businesses can benefit from AI while maintaining appropriate human oversight and responsible practices.
"""

print("SELF-REFLECTION AND IMPROVEMENT")
print("=" * 70)
print(reflection_output)

SELF-REFLECTION AND IMPROVEMENT

CRITIQUE
Accuracy: The original summary is generally accurate, but the statement that AI helps businesses make better decisions is broad and should be presented more carefully.

Clarity: The ideas are easy to understand, but some wording is general and could be more specific.

Audience: The summary is appropriate for a college MIS student, but it can use slightly more academic wording.

Length: The original summary is shorter than the required 70-100 words.

Organization: The ideas can be organized more clearly by discussing business uses first and responsible use second.

Focus: The main focus is appropriate, but the importance of accuracy, privacy, and responsible use can be explained more clearly.

REVISED SUMMARY:
Artificial intelligence is increasingly used by businesses to automate tasks, analyze information, and support decision-making. These applications can improve efficiency and help organizations work with large amounts of data. However, busi

## Before and After Comparison

The original summary is compared with the revised summary
to demonstrate improvement based on the self-critique.

In [31]:
print("=" * 70)
print("BEFORE — ORIGINAL SUMMARY")
print("=" * 70)
print(original_summary)

print("\n" + "=" * 70)
print("AFTER — AI SELF-REFLECTION AND REVISED SUMMARY")
print("=" * 70)
print(reflection_output)

BEFORE — ORIGINAL SUMMARY

Artificial intelligence is becoming more common in business.
Companies use AI to automate tasks and analyze information.
AI can help businesses save time and make better decisions.
However, businesses should also consider accuracy, privacy,
and how employees use AI.


AFTER — AI SELF-REFLECTION AND REVISED SUMMARY

CRITIQUE
Accuracy: The original summary is generally accurate, but the statement that AI helps businesses make better decisions is broad and should be presented more carefully.

Clarity: The ideas are easy to understand, but some wording is general and could be more specific.

Audience: The summary is appropriate for a college MIS student, but it can use slightly more academic wording.

Length: The original summary is shorter than the required 70-100 words.

Organization: The ideas can be organized more clearly by discussing business uses first and responsible use second.

Focus: The main focus is appropriate, but the importance of accuracy, privac

## Demonstrated Improvement

The original summary provides a basic explanation of business
AI use but uses broad statements and has limited detail.

The self-reflection prompt requires the AI to evaluate accuracy,
clarity, audience, length, organization, and focus before revising.

The revised version is designed to be clearer, more organized,
appropriate for a college MIS audience, and within the required
70-100 word length.

## Exercise 3 Summary

The self-reflection process first evaluates the original summary
against explicit criteria and then creates a revised version.

This demonstrates prompt engineering through clear instructions,
specific constraints, audience awareness, and goal alignment.

The before-and-after outputs provide evidence that the summary
was reviewed and improved.